Aqui vamos testar codigos para depois criar a pipeline 

In [2]:
# imports
import json
import requests
from dotenv import load_dotenv
import os

In [18]:
# carregando a chave da api
load_dotenv()

API_KEY = os.getenv('OPENWEATHER_API_KEY')
if not API_KEY:
    raise ValueError('A chave da api "OPENWEATHER_API_KEY" não foi encontrada. Verifique o arquivo')
print(API_KEY)

10bb80f9b0c208099ef0028a466fcd49


In [19]:
city = 'São Paulo'

def get_previsao_atual(city, api_key):
    # Extrair dados metereologicos da cidade selecionada
    
    base_url = 'https://api.openweathermap.org/data/2.5/weather'
    params = {
    'q': city,
    'appid':api_key,
    'units': 'metric',
    'lang': 'pt_br'
    }
    try:
        response = requests.get(base_url, params= params)
        response.raise_for_status()
        data = response.json()
        dados_prev_atual = {}
        
        cidade_atual = data['name']
        temperatura_atual = data['main']['temp']
        temp_max = data['main']['temp_max']
        temp_min = data['main']['temp_min']
        sensacao_termica = data['main']['feels_like']

        dados_prev_atual = {
            'Cidade': cidade_atual,
            'Temperatura Atual': temperatura_atual,
            'Minima': temp_min,
            'Maxima': temp_max,
            'Sensação Termica': sensacao_termica
        }

        return dados_prev_atual
    except requests.exceptions.RequestException as e:
        print(f'Erro ao obter o clima atual para {city}: {e}')
        return None

    

In [21]:
print(API_KEY)
get_previsao_atual(city,API_KEY)


10bb80f9b0c208099ef0028a466fcd49
Erro ao obter o clima atual para São Paulo: 401 Client Error: Unauthorized for url: https://api.openweathermap.org/data/2.5/weather?q=S%C3%A3o+Paulo&appid=10bb80f9b0c208099ef0028a466fcd49&units=metric&lang=pt_br


In [46]:
def buscar_previsao_futura(city,api_key):
    base_url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = params = {
    'q': city,
    'appid':api_key,
    'units': 'metric',
    'lang': 'pt_br'
    }
    try:
        response = requests.get(base_url, params)
        response.raise_for_status()
        return response.json().get('list', [])
    except requests.exceptions.RequestException as e:
        print(f'Erro ao buscar a previsão futura para {city}: {e}')
        return None

buscar_previsao_futura(city, API_KEY)

Erro ao buscar a previsão futura para São Paulo: 401 Client Error: Unauthorized for url: https://api.openweathermap.org/data/2.5/forecast?q=S%C3%A3o+Paulo&appid=10bb80f9b0c208099ef0028a466fcd49&units=metric&lang=pt_br


In [19]:
from datetime import date, timedelta, datetime
def transoformar_dados_previsao(lista_previsao):
    # Transforma dados e busca as previsoes para os proximos 3 dias usando a api e a bibilioteca datetime
    if not lista_previsao:
        return []
    hoje = date.today()
    limite_data = hoje + timedelta(days=4)
    previsoes_diarias = {}

    for previsao in lista_previsao:
        timestamp = datetime.strptime(previsao['dt_txt'],'%Y-%m-%d %H:%M:%S')
        data_previsao = timestamp.date()

        if hoje < data_previsao < limite_data:
            temp_atual = previsao['main']['temp']

            if data_previsao not in previsoes_diarias:
                previsoes_diarias[data_previsao] = {
                    'data': data_previsao.strftime('%Y-%m-%d'),
                    'temp_max':temp_atual,
                    'clima_representativo': previsao['weather'][0]['description']
                }
            else:
                if temp_atual > previsoes_diarias[data_previsao]['temp_max']:
                    previsoes_diarias[data_previsao]['temp_max'] = temp_atual
                    previsoes_diarias[data_previsao]['clima_representativo'] = previsao['weather'][0]['description']
                
    return list(previsoes_diarias.values())


In [20]:
lista_bruta = buscar_previsao_futura(city, API_KEY)
lista_previsao = transoformar_dados_previsao(lista_bruta)

print(lista_previsao)
#print(lista_bruta)

Erro ao buscar a previsão futura para São Paulo: 401 Client Error: Unauthorized for url: https://api.openweathermap.org/data/2.5/forecast?q=S%C3%A3o+Paulo&appid=9e540c3569fdd15dfe79de7347959931&units=metric&lang=pt_br
[]
